# Самостоятельная работа 03. Градиентный спуск шаг за шагом

Цель: написать градиентный спуск с нуля из трёх маленьких функций — градиент, шаг, цикл — и убедиться, что он приходит к той же прямой, что и точные формулы.

Весь градиентный спуск — это одна формула, повторённая много раз:

$$\theta_{\text{след}} = \theta - \eta \nabla \text{MSE}(\theta)$$

Здесь $\theta = (a, b)$ — параметры прямой, $\nabla \text{MSE}$ — градиент ошибки, $\eta$ — скорость обучения (на слайдах лекции она обозначена $\alpha$). Вы соберёте формулу из трёх кирпичиков, и каждое задание вызывает функцию из предыдущего:

| Кирпичик | Что делает | Задание |
|---|---|---|
| `mse_gradient` | считает, куда и насколько круто растёт ошибка | 1 |
| `gradient_step` | делает один шаг против градиента — вызывает `mse_gradient` | 2 |
| `gradient_descent` | повторяет шаг много раз — вызывает `gradient_step` | 3 |

Правила:

- задания выстроены цепочкой, поэтому защищаются по порядку: защитить можно любое число заданий, но подряд, начиная с первого; чтобы защитить задание 2, должно работать задание 1, а для задания 3 — задания 1 и 2;
- уйти с занятия раньше времени можно только после защиты хотя бы одного задания;
- баллы ставятся только за защищённые задания: каждое — 1 балл; выполненное, но не защищённое задание даёт 0;
- **бонус:** кто выполнит и защитит все три задания на этом же занятии, получает ещё 3 балла сверху — всего 6;
- на защите нужно объяснить любую строку своего кода и при преподавателе изменить его;
- ИИ-помощником можно пользоваться точечно — вспомнить функцию, разобраться с ошибкой; код, который вы не можете объяснить, не засчитывается;
- перед защитой перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и задания по порядку — самопроверки защищаемых заданий должны пройти.

Материалы:

- лекция 04 «Обучение линейной регрессии», раздел «Градиентный спуск»: касательная, шаг против производной, градиент, скорость обучения;
- Орельен Жерон, «Прикладное машинное обучение с помощью Scikit-Learn и TensorFlow», гл. 4, стр. 160–166;
- блокнот практического занятия 05 «Обучение линейной регрессии: МНК и градиентный спуск»;
- блокнот `02-linear-regression.ipynb`, задание 1: там вы уже брали частные производные той же функции ошибки.

| № | Что сделать | Тип | Баллы |
|---|---|---|---|
| 1 | Градиент: функция `mse_gradient` | базовое | 1 |
| 2 | Шаг: функция `gradient_step` и три шага вручную | базовое | 1 |
| 3 | Цикл: функция `gradient_descent` и скорость обучения | базовое | 1 |
| | Все три задания защищены на этом же занятии | бонус | +3 |

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.version, np.__version__, pd.__version__, matplotlib.__version__

## Данные

Задача та же, что в блокноте `02-linear-regression.ipynb`: служба такси оценивает время поездки по расстоянию прямой $\hat{y} = a x + b$, где $x$ — расстояние, км, а $\hat{y}$ — время поездки, мин. Функция ошибки — $\text{MSE}(a, b) = \frac{1}{m} \sum_{i=1}^{m} (a x_i + b - y_i)^2$. Точные формулы дали там $a \approx 2{,}22$ и $b \approx 2{,}84$ — ваш спуск должен прийти туда же, но шагами.

Ячейки ниже готовы — просто выполните их. Функция `line_mse` считает MSE прямой, функция `is_minimum` нужна самопроверкам: она проверяет, что небольшой сдвиг $a$ или $b$ в любую сторону не уменьшает ошибку.

In [ ]:
def make_trips(n_trips, seed=0):
    """Генерирует поездки такси: расстояние и время в пути.

    Args:
        n_trips: Число поездок.
        seed: Зерно генератора случайных чисел.

    Returns:
        Массивы расстояний, км, и времени поездки, мин.
    """
    rng = np.random.default_rng(seed)
    distance = rng.uniform(0.5, 15, size=n_trips)
    duration = 3 + 2.2 * distance + rng.normal(0, 3, size=n_trips)  # Пробки дают разброс в несколько минут
    return distance, duration


def line_mse(a, b, x, y):
    """Считает MSE прямой y = a·x + b на точках (x, y).

    Args:
        a: Наклон прямой.
        b: Свободный член.
        x: Значения признака.
        y: Целевые значения.

    Returns:
        Среднеквадратичная ошибка.
    """
    return np.mean((y - a * x - b) ** 2)


def is_minimum(a, b, x, y, shift=0.01):
    """Проверяет, что сдвиг a или b на shift в любую сторону не уменьшает MSE.

    Args:
        a: Наклон прямой.
        b: Свободный член.
        x: Значения признака.
        y: Целевые значения.
        shift: Величина сдвига параметра.

    Returns:
        True, если ни один из четырёх сдвигов не уменьшает MSE.
    """
    best_mse = line_mse(a, b, x, y)

    for shift_a, shift_b in [(shift, 0), (-shift, 0), (0, shift), (0, -shift)]:
        if line_mse(a + shift_a, b + shift_b, x, y) < best_mse:
            return False

    return True


distance, duration = make_trips(n_trips=100, seed=42)
distance.shape, duration[:5].round(1)

In [ ]:
plt.scatter(distance, duration, s=12)
plt.xlabel('Расстояние, км')
plt.ylabel('Время поездки, мин')
plt.show()

## Задание 1. Градиент: куда и насколько круто растёт ошибка (базовое, 1 балл)

Спуск начинается с вопроса: если чуть увеличить $a$ (или $b$), ошибка вырастет или упадёт — и насколько быстро? Ответ даёт частная производная: её знак показывает, в какую сторону ошибка растёт, а модуль — насколько крут склон. Обе частные производные вместе — это градиент.

**Дано:** массивы `distance` (значения $x_i$) и `duration` (значения $y_i$).

**Получить:**

- функцию `mse_gradient(a, b, x, y)`, которая возвращает кортеж `(grad_a, grad_b)` — частные производные MSE по $a$ и по $b$ в точке $(a, b)$;
- `grad_a_start`, `grad_b_start` — градиент в точке $a = 0$, $b = 0$, то есть для горизонтальной прямой $\hat{y} = 0$.

**Как действовать:**

1. Посмотрите на функцию ошибки $\text{MSE}(a, b) = \frac{1}{m} \sum_{i=1}^{m} (a x_i + b - y_i)^2$. Выражение в скобках — остаток: насколько прямая промахивается по поездке $i$.
2. Производная по $b$ дана готовой: $\partial \text{MSE} / \partial b = \frac{2}{m} \sum_{i=1}^{m} (a x_i + b - y_i)$ — внешняя функция $u^2$ даёт $2u$, а производная остатка по $b$ равна 1 (правило цепочки). Так же выведите $\partial \text{MSE} / \partial a$: чему равна производная остатка по $a$?
3. Запишите обе производные в `mse_gradient` на NumPy без цикла по поездкам: сначала массив остатков, затем средние — `np.mean` и есть $\frac{1}{m} \sum$.
4. Вызовите функцию в точке $a = 0$, $b = 0$.

Конечные разности и scikit-learn здесь не используйте: градиент считается по формулам, которые вы вывели. Это те же частные производные, что в задании 1 блокнота `02-linear-regression.ipynb`, только для MSE и без приравнивания к нулю: теперь мы подставляем текущие $a$ и $b$ и получаем два числа.

**Что почитать:**

- лекция 04, слайды «Градиент: производные по всем параметрам» и «Градиент нашей функции»; книга, стр. 164;
- [Частная производная](https://ru.wikipedia.org/wiki/Частная_производная), [Дифференцирование сложной функции](https://ru.wikipedia.org/wiki/Дифференцирование_сложной_функции) (правило цепочки) и [Градиент](https://ru.wikipedia.org/wiki/Градиент) — Википедия;
- [Кортежи](https://docs.python.org/3.8/tutorial/datastructures.html#tuples-and-sequences) — как вернуть из функции два числа сразу;
- [`np.mean`](https://numpy.org/doc/1.24/reference/generated/numpy.mean.html) — среднее элементов массива.

In [ ]:
def mse_gradient(a, b, x, y):
    """Считает градиент MSE прямой y = a·x + b по параметрам a и b.

    Args:
        a: Наклон прямой.
        b: Свободный член.
        x: Значения признака.
        y: Целевые значения.

    Returns:
        Кортеж (grad_a, grad_b): частные производные MSE по a и по b.
    """
    ...  # Ваш код


grad_a_start, grad_b_start = ..., ...  # Градиент в точке a = 0, b = 0

grad_a_start, grad_b_start

Самопроверка сравнивает ваш градиент с численной производной — определением производной со слайда «Откуда берётся касательная», где предел заменён малым $\varepsilon$: $f'(a) \approx \frac{f(a + \varepsilon) - f(a - \varepsilon)}{2 \varepsilon}$. Если формула верна, числа совпадут в любой точке. Так проверяют градиенты и авторы настоящих библиотек — подробнее в статье [Численное дифференцирование](https://ru.wikipedia.org/wiki/Численное_дифференцирование).

In [ ]:
eps = 1e-6

for a_test, b_test in [(0.0, 0.0), (1.5, -2.0), (3.0, 4.0)]:
    analytic_a, analytic_b = mse_gradient(a_test, b_test, distance, duration)
    numeric_a = (line_mse(a_test + eps, b_test, distance, duration)
                 - line_mse(a_test - eps, b_test, distance, duration)) / (2 * eps)
    numeric_b = (line_mse(a_test, b_test + eps, distance, duration)
                 - line_mse(a_test, b_test - eps, distance, duration)) / (2 * eps)
    assert np.isclose(analytic_a, numeric_a, rtol=1e-4), f'Производная по a неверна в точке ({a_test}, {b_test})'
    assert np.isclose(analytic_b, numeric_b, rtol=1e-4), f'Производная по b неверна в точке ({a_test}, {b_test})'

assert np.allclose([grad_a_start, grad_b_start], mse_gradient(0.0, 0.0, distance, duration)), (
    'grad_a_start и grad_b_start — это градиент в точке a = 0, b = 0')

'Задание 1: самопроверка пройдена'

Оба числа `grad_a_start` и `grad_b_start` отрицательны. Что это значит: в какую сторону нужно сдвинуть $a$ и $b$, чтобы ошибка уменьшилась? Почему `grad_a_start` по модулю почти в десять раз больше, чем `grad_b_start`?

*Ваш ответ:* …

## Задание 2. Один шаг вниз (базовое, 1 балл)

Градиент показывает, куда ошибка растёт быстрее всего, — значит, чтобы ошибка падала, нужно шагнуть в противоположную сторону. Длину шага задаёт скорость обучения $\eta$: маленькая — осторожные шаги, большая — прыжки.

**Дано:** функция `mse_gradient` из задания 1.

**Получить:**

- функцию `gradient_step(a, b, x, y, learning_rate)`, которая возвращает кортеж `(a, b)` — параметры после одного шага;
- `first_steps` — таблицу `pd.DataFrame` из 4 строк: старт $a = 0$, $b = 0$ и точки после каждого из трёх шагов со скоростью обучения 0,01; столбцы `a`, `b`, `grad_a`, `grad_b`, `mse` — параметры, градиент и MSE в этой точке, как в таблице «Три шага вручную» со слайда лекции.

**Как действовать:**

1. Внутри `gradient_step` получите градиент вызовом `mse_gradient` — формулы заново не пишите: в этом и смысл цепочки.
2. Сдвиньте оба параметра против градиента: $a_{\text{след}} = a - \eta \cdot \partial \text{MSE} / \partial a$ и $b_{\text{след}} = b - \eta \cdot \partial \text{MSE} / \partial b$. Оба сдвига считаются по градиенту в текущей точке, то есть до шага.
3. Не запуская код, ответьте на вопрос-предсказание ниже.
4. Начните с $a = 0$, $b = 0$ и трижды сделайте шаг со скоростью обучения 0,01. Для старта и после каждого шага запишите строку таблицы: $a$, $b$, градиент из `mse_gradient` и MSE из `line_mse`. Удобно копить строки словарями в списке и в конце превратить список в `pd.DataFrame`.

**Что почитать:**

- лекция 04, слайды «Шаг против производной», «Шаг градиентного спуска» и «Три шага вручную»; книга, стр. 164–165;
- [Градиентный спуск](https://ru.wikipedia.org/wiki/Градиентный_спуск) — Википедия;
- [`pd.DataFrame`](https://pandas.pydata.org/pandas-docs/version/2.0/reference/api/pandas.DataFrame.html) — таблица из списка словарей: ключи словаря становятся столбцами.

Предсказание: по своему `grad_a_start` и скорости обучения 0,01 оцените, куда и примерно на сколько сдвинется $a$ на первом шаге. Окажется ли $a$ после шага больше 2,22 — наклона, который дали точные формулы?

*Ваш ответ:* …

In [ ]:
def gradient_step(a, b, x, y, learning_rate):
    """Делает один шаг градиентного спуска по MSE.

    Args:
        a: Наклон прямой до шага.
        b: Свободный член до шага.
        x: Значения признака.
        y: Целевые значения.
        learning_rate: Скорость обучения.

    Returns:
        Кортеж (a, b): параметры прямой после шага.
    """
    ...  # Ваш код


first_steps = ...  # Ваш код: старт и три шага со скоростью обучения 0,01

first_steps

In [ ]:
a_moved, b_moved = gradient_step(1.0, 1.0, distance, duration, learning_rate=0.001)
grad_a_here, grad_b_here = mse_gradient(1.0, 1.0, distance, duration)

assert np.allclose(gradient_step(1.0, 1.0, distance, duration, learning_rate=0.0), [1.0, 1.0]), (
    'При скорости обучения 0 шаг не должен сдвигать параметры')
assert np.allclose([a_moved - 1.0, b_moved - 1.0], [-0.001 * grad_a_here, -0.001 * grad_b_here]), (
    'Сдвиг должен быть против градиента, посчитанного до шага, и пропорционален скорости обучения')
assert {'a', 'b', 'grad_a', 'grad_b', 'mse'} <= set(first_steps.columns), 'Нужны столбцы a, b, grad_a, grad_b, mse'
assert len(first_steps) == 4, 'В таблице 4 строки: старт и три шага'
assert np.allclose(first_steps.iloc[0][['a', 'b']], [0.0, 0.0]), 'Спуск начинается с a = 0, b = 0'
assert np.allclose(first_steps.iloc[1][['a', 'b']], gradient_step(0.0, 0.0, distance, duration, 0.01)), (
    'Строка 1 — точка после первого шага со скоростью обучения 0,01')
assert (first_steps['mse'].diff().dropna() < 0).all(), 'MSE должна уменьшаться с каждым шагом'

'Задание 2: самопроверка пройдена'

По своей таблице `first_steps`: почему $a$ прыгает то выше 2,22, то ниже, а знак `grad_a` меняется на каждом шаге? Почему $b$ за три шага почти не сдвинулся к 2,84?

*Ваш ответ:* …

## Задание 3. Цикл: спуск до дна (базовое, 1 балл)

Один шаг немного уменьшает ошибку. Повторите его много раз — это и есть весь градиентный спуск.

**Дано:** функция `gradient_step` из задания 2, функция `line_mse`.

**Получить:**

- функцию `gradient_descent(x, y, learning_rate, n_steps)`, которая возвращает `a`, `b` и список `mse_history` — значение MSE после каждого шага;
- `a_gd`, `b_gd`, `mse_history` — результат запуска со скоростью обучения 0,01 на 2000 шагов и график `mse_history`;
- `mse_history_fast` — историю MSE запуска со скоростью обучения 0,02 на 200 шагов.

**Как действовать:**

1. Начните с $a = 0$, $b = 0$ и пустого списка `mse_history`.
2. В цикле на `n_steps` повторений вызывайте `gradient_step`: градиент и шаг уже написаны в заданиях 1 и 2, поэтому в цикле нужны только вызов шага и запись MSE после него в `mse_history`.
3. Запустите функцию со скоростью обучения 0,01 на 2000 шагов и постройте график `mse_history`: по оси $x$ — номер шага, по оси $y$ — MSE в логарифмическом масштабе, иначе после первых шагов кривая сольётся с осью.
4. **Предскажи, потом проверь.** Не запуская код, запишите в ячейку ответа, что произойдёт со спуском при скорости обучения 0,02 и почему. Затем запустите функцию с `learning_rate=0.02` на 200 шагов и сравните историю MSE с предсказанием.

**Что почитать:**

- лекция 04, слайды «После 40 шагов» и «Скорость обучения: мала, в самый раз, велика»; книга, стр. 160–166;
- [Оператор `for`](https://docs.python.org/3.8/tutorial/controlflow.html#for-statements) — цикл с заданным числом повторений через `range`;
- [`plt.plot`](https://matplotlib.org/3.7.5/api/_as_gen/matplotlib.pyplot.plot.html) и [`plt.yscale`](https://matplotlib.org/3.7.5/api/_as_gen/matplotlib.pyplot.yscale.html) — линия и логарифмическая шкала оси.

In [ ]:
def gradient_descent(x, y, learning_rate, n_steps):
    """Обучает прямую y = a·x + b пакетным градиентным спуском по MSE.

    Args:
        x: Значения признака.
        y: Целевые значения.
        learning_rate: Скорость обучения.
        n_steps: Число шагов.

    Returns:
        Кортеж (a, b, mse_history): параметры прямой и MSE после каждого шага.
    """
    ...  # Ваш код


a_gd, b_gd, mse_history = gradient_descent(distance, duration, learning_rate=0.01, n_steps=2000)
a_gd, b_gd, mse_history[-1]

In [ ]:
...  # Ваш код: график mse_history, ось y в логарифмическом масштабе

Предсказание: что произойдёт со спуском при скорости обучения 0,02 и почему?

*Ваш ответ:* …

In [ ]:
mse_history_fast = ...  # Ваш код: 200 шагов со скоростью обучения 0,02

mse_history_fast[:3], mse_history_fast[-1]

In [ ]:
a_three, b_three, _ = gradient_descent(distance, duration, learning_rate=0.01, n_steps=3)

assert len(mse_history) == 2000, 'В истории запуска на 2000 шагов должно быть 2000 значений MSE'
assert is_minimum(a_gd, b_gd, distance, duration), 'Спуск не пришёл в минимум ошибки: проверьте цикл и шаг'
assert np.allclose([a_three, b_three], first_steps.iloc[3][['a', 'b']]), (
    'Три шага цикла должны дать ту же точку, что последняя строка first_steps из задания 2')
assert len(mse_history_fast) == 200, 'Нужна история MSE 200 шагов со скоростью обучения 0,02'

'Задание 3: самопроверка пройдена'

Совпало ли предсказание с результатом? Объясните, почему при 0,01 спуск сходится, а при 0,02 — нет. Почему на вашем графике MSE сначала падает круто, а потом почти горизонтально?

*Ваш ответ:* …

## Итог: как прямая шла к ответу

Ячейка ниже готова: она рисует прямые из вашей таблицы `first_steps` и прямую после 2000 шагов и сравнивает ответ спуска с точной формулой. Она выполнится, когда сделаны все три задания.

In [ ]:
x_line = np.array([0.0, 15.0])
plt.scatter(distance, duration, s=12, color='lightgray')

for step, row in first_steps.iterrows():
    plt.plot(x_line, row['a'] * x_line + row['b'], '--', label=f'шаг {step}: a = {row["a"]:.2f}, b = {row["b"]:.2f}')

plt.plot(x_line, a_gd * x_line + b_gd, color='black', label=f'шаг 2000: a = {a_gd:.2f}, b = {b_gd:.2f}')
plt.xlabel('Расстояние, км')
plt.ylabel('Время поездки, мин')
plt.legend()
plt.show()

a_formula, b_formula = np.polyfit(distance, duration, deg=1)
print(f'Градиентный спуск: a = {a_gd:.4f}, b = {b_gd:.4f}')
print(f'Точная формула:    a = {a_formula:.4f}, b = {b_formula:.4f}')

## Готово к защите

Перед тем как звать преподавателя:

- перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и задания по порядку — самопроверки защищаемых заданий должны пройти;
- убедитесь, что можете объяснить каждую строку своего кода, показать, где меняются формула градиента, скорость обучения, число шагов и начальная точка, и заранее сказать, что будет после такой правки.